In [14]:
import torch 
import src.util_tfrecord as util_tfrecords
from pathlib import Path
import tensorflow as tf
import numpy as np 
import os
from pathlib import Path
import importlib
from IPython.display import Audio, display

import src.backbone_lightning as lightning_module 
import yaml
from pytorch_lightning import Trainer, seed_everything

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


In [15]:
config_path = "config/binaural_attn/word_task_v10_backbone_word_config_saddler_dataset.yaml"
# config_path = "config/binaural_attn/word_task_half_co_loc_v06.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 2
config['hparas']['batch_size'] = 32
print(config['hparas'])

{'valid_step': 2000, 'epochs': 1000, 'optimizer': 'AdamW', 'lr': 0.0001, 'eps': 1e-07, 'gradient_clip_val': 1, 'gradient_clip_algorithm': 'norm', 'batch_size': 32, 'mask_cues': False}


In [16]:
module = lightning_module.BinauralBackBoneModule(config)


num_classes={'num_words': 801}
Model performing word task
Conv block order: LN -> Conv -> ReLU
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [17]:
train_set = module.train_dataloader()

In [18]:
batch = next(iter(train_set))

In [19]:
batch['sr']

<tf.Tensor: shape=(32,), dtype=int64, numpy=
array([44100, 44100, 44100, 44100, 44100, 44100, 44100, 44100, 44100,
       44100, 44100, 44100, 44100, 44100, 44100, 44100, 44100, 44100,
       44100, 44100, 44100, 44100, 44100, 44100, 44100, 44100, 44100,
       44100, 44100, 44100, 44100, 44100])>

In [20]:
sound = batch['signal']

In [21]:
Audio(sound[31].numpy().T, rate=44100, normalize=False)

In [22]:
sound = torch.from_numpy(batch['signal'].numpy())
b, time, channels = sound.shape
sound = torch.permute(sound, (0, 2, 1))  # (b, channels, time)


In [23]:
Audio(sound[31], rate=44100, normalize=False)

In [24]:
batch.keys()

dict_keys(['client_id_int', 'dbspl', 'gender_int', 'index', 'label_background_int', 'label_env_int', 'label_loc_foreground_azim', 'label_loc_foreground_dist', 'label_loc_foreground_elev', 'label_talker_int', 'label_word_int', 'signal', 'snr', 'split_int', 'sr', 'talker_int', 'word_int'])

In [25]:
word_label = batch['label_word_int']
word_label

<tf.Tensor: shape=(32,), dtype=int64, numpy=
array([335, 632, 473,   0, 281,   0,   0, 308,   0,   0,   0,  23,   0,
         0,   0,   0,   0, 466, 755,   0, 285,   0, 710,   0,   0, 425,
         0, 639, 569,   0, 789, 298])>

In [26]:
seed_everything(0)
importlib.reload(lightning_module)

module = lightning_module.BinauralBackBoneModule(config)
trainer = Trainer(
    precision="32",
    limit_val_batches=0.0,
    num_nodes=1,
    # benchmark=True,
    devices=1, # was gpus=1,
    # detect_anomaly=True,
    # strategy="ddp_notebook",
    gradient_clip_val=1, # config['hparas']['gradient_clip_val'],
    gradient_clip_algorithm='norm', # config['hparas'].get('gradient_clip_algorithm', 'norm'),
    accelerator="gpu",
)
trainer.fit(module)

[rank: 0] Seed set to 0


num_classes={'num_words': 801}
Model performing word task
Conv block order: LN -> Conv -> ReLU


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type                         | Params
----------------------------------------------------------------
0 | model          | BackBoneCNN                  | 62.6 M
1 | coch_gram      | AttnAudioInputRepresentation | 0     
2 | word_loss_fn   | CrossEntropyLoss             | 0     
3 | train_acc      | ModuleDict                   | 0     
4 | valid_acc      | ModuleDict                   | 0     
5 | test_acc       | ModuleDict                   | 0     
6 | test_confusion | ModuleDict                   | 0     
----------------------------------------------------------------
62.6 M    Trainable params
0         Non-trainable params
62.6 M    Total params
250.492   Total estimated model params size (MB)


center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram
Epoch 0: |          | 64/? [01:11<00:00,  0.89it/s, v_num=4.17e+7, train_loss=4.480, grad_norm=15.00]